# Õppetund 02 - Microsoft Agendi Raamistiku Uurimine

**Microsoft Agendi Raamistik (MAF)** on ühtne raamistik tehisintellekti agentide loomiseks. See pakub puhta ja komposiitset arhitektuuri nelja põhielemendiga:

- **Kliendi** – ühendub tehisintellekti mudeli lõpp-punktiga ja haldab suhtlust
- **Agent** – katab kliendi juhiste ja tööriistade definitsioonidega
- **Tööriistad** – laiendavad agendi võimekust kohandatud funktsioonidega, mida mudel saab kutsuda
- **Sessioon** – hoiab vestluse ajalugu mitme sammu pikkuse suhtluse jaoks

Selles õppetükis ehitame **reisibroneerimise agendi**, mis kontrollib sihtkoha saadavust, kasutades neid kontseptsioone.


## Seadistamine


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Agent Raamistiku Arhitektuuri Mõistmine

Microsoft Agent Framework järgib kihilist arhitektuuri:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Kliendi pool** – `FoundryChatClient` ühendub Azure OpenAI juurutusega. See haldab autentimist, päringu vormindamist ja vastuse analüüsimist.
2. **Agent** – Luuakse kliendi abil `provider.create_agent()` kaudu, agent ühendab mudelile juurdepääsu koos juhistega (süsteemiprompt) ja tööriistadega.
3. **Tööriistad** – Python funktsioonid, mis on tähistatud `@tool` dekooraatoriga, mida agent saab kutsuda toimingute tegemiseks või andmete hankimiseks.
4. **Sessioon** – `AgentSession` objekt (loodud `agent.create_session()` abil), mis salvestab vestluse ajaloo võimaldades mitmekordset dialoogi, kus agent mäletab varasemat konteksti.

Loome iga kihi samm-sammult.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Tööriistade lisamine @tool dekoorijaga

Tööriistad võimaldavad agentidel teha toiminguid, mis ületavad teksti genereerimise. `@tool` dekoorija teisendab tavapärase Pythoni funktsiooni millegagi, mida agent saab kutsuda.

Olulised punktid:
- Kasutage `Annotated[type, "kirjeldus"]`, et mudel mõistaks iga parameetrit.
- Docstring muutub tööriista kirjelduseks, mida mudel näeb.
- `approval_mode="never_require"` tähendab, et tööriist töötab automaatselt ilma kasutaja kinnitust nõudmata.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Agendi loomine tööriistadega

Nüüd ühendame kliendi, juhised ja tööriistad agendiks. `instructions` toimivad süsteemi käsuna — need määratlevad agendi isiksuse ja käitumise.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Mitme-käigulised vestlused sessioonidega

`AgentSession` (loodud `agent.create_session()` abil) jälgib kõiki sõnumeid vestluse jooksul. Kui edastada sama sessioon iga `agent.run()` väljakutse juurde, saab agent ligipääsu kogu vestluse ajaloole ja võib viidata varasematele sõnumitele.

Me edastame `tools=[check_destination_availability]`, et agent saaks igal sammul kutsuda meie saadavuse kontrollijat.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Kokkuvõte

Selles õppetükis uurisite Microsoft Agent raamistiku nelja alustala:

| Mõiste | Mida sa õppisid |
|---------|------------------|
| **Kliendi** | `FoundryChatClient` ühendub Azure OpenAI-ga volitustepõhise autentimisega |
| **Agent** | `provider.create_agent()` ühendab mudeli ühenduse juhiste ja nimega |
| **Tööriistad** | `@tool` dekoratsioon avaldab Python funktsioonid agendi jaoks kutsumiseks |
| **Sessioon** | `agent.create_session()` haldab vestluse ajalugu mitme vooru jooksul |

Need ehituskivid moodustavad koos agendid, kes suudavad pidada loomulikke vestlusi, kutsuda väliseid funktsioone ja säilitada konteksti — aluse keerukamate agendimustrite jaoks hilisemates õppetundides.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
